# Shida clinical correlations and gait comparisons: FDR correction

This companion notebook applies multiple-testing correction to **two complete internal-cohort hypothesis families**: 396 clinical correlations (18 NRMSE metrics × 22 clinical measures) and 19 gait-parameter comparisons. Shida is the internal cohort. Viglialoro is external and is not pooled into these families.

**Current status: corrected results cannot yet be calculated.** The two supplied participant workbooks are embedded below, but neither contains the original test p-values, participant-level model errors, or the 19 gait-parameter observations. This notebook does not turn missing test results into zero discoveries.

Run this in a **separate Colab session with a CPU runtime**. It does not change or launch model training. It saves outputs separately from `complete_studies/reviewer_complete_v1`.

1. Run setup and inspect the metadata audit.
2. Supply the complete unrounded test tables using the CSV templates, or export existing results into the documented columns.
3. Run correction and download the results. A missing family remains explicitly pending.

For new nested-LOSO results, wait for complete participant-level held-out predictions and regenerate the tests. Do not combine old-analysis and new-analysis p-values. The list of 22 clinical variables, medication state, error aggregation and 19 gait tests must match the analysis being corrected.

## Statistical specification

Benjamini–Hochberg (BH) correction is applied across **all 396 correlations together**, and separately across **all 19 gait comparisons**. The significance threshold is q < 0.05. The columns `q_bh` and `q_by` are FDR-adjusted p-values, not Storey's estimated q-values.

BH assumes independence or the relevant positive-dependence condition. Correlated model errors and clinical variables do not automatically establish that condition. The notebook therefore also reports the more conservative Benjamini–Yekutieli (BY) correction, valid under arbitrary dependence of valid p-values, as a sensitivity analysis. If dependence assumptions cannot be defended, base confirmatory statements on BY. Both columns are always reported; do not choose a method after seeing which gives more discoveries.

Correction cannot repair invalid underlying tests or treating cycles as independent participants. Use participant-level observations and report the sample size for each test. Do not correct just the 62 previously nominally significant correlations or just the 11 reported gait findings. For missing/undefined tests, revisit and document the testing family instead of silently dropping rows or substituting p = 1.

The expectation under the global null is 396 × 0.05 = 19.8 nominal findings. This does not establish that exactly 20 of the observed 62 are false and is not an adjustment formula.

Reference: [SciPy false_discovery_control](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.false_discovery_control.html), including Benjamini & Hochberg (1995) and Benjamini & Yekutieli (2001).

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, hashlib, json, platform, re, shutil
import numpy as np
import pandas as pd
import scipy
from IPython.display import display

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/Shida_Nested_LOSO')
else:
    PROJECT = Path('/content/Shida_Nested_LOSO')
ROOT = PROJECT / 'fdr_postprocessing'
SOURCES = ROOT / 'sources'
INPUTS = ROOT / 'inputs'
TEMPLATES = ROOT / 'templates'
for folder in (SOURCES, INPUTS, TEMPLATES):
    folder.mkdir(parents=True, exist_ok=True)
print('FDR files:', ROOT)
print('No GPU is needed. No training experiment will be modified.')

In [ ]:
EMBEDDED = {'PDGinfo_Shida.xlsx': {'sha256': '7c85667796fb2b127da23e762230b7539e0fa1c1f253b0a049d2d01e6c20e229', 'base64': '...'}, 'Partecipants_Viglialoro.xlsx': {'sha256': '7c74ed83dc63d8da0252eeb380073b730c6a40d020ab0727fcf1228e3e7c23f7', 'base64': '...'}}
for name, spec in EMBEDDED.items():
    content = base64.b64decode(spec['base64'])
    assert hashlib.sha256(content).hexdigest() == spec['sha256']
    path = SOURCES / name
    if path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() != spec['sha256']:
        raise ValueError(f'Existing source differs: {path}. Preserve it and use a separate source folder.')
    if not path.exists():
        path.write_bytes(content)
print('Both original participant workbooks restored and verified.')

In [ ]:
WAVEFORM_AUDIT_CSV = 'task,subject,n_cycles\nAP_Left,SUB01,7\nAP_Left,SUB02,23\nAP_Left,SUB03,9\nAP_Left,SUB05,2\nAP_Left,SUB06,21\nAP_Left,SUB07,1\nAP_Left,SUB10,20\nAP_Left,SUB11,19\nAP_Left,SUB12,8\nAP_Left,SUB13,12\nAP_Left,SUB14,21\nAP_Left,SUB16,14\nAP_Left,SUB17,11\nAP_Left,SUB18,20\nAP_Left,SUB19,18\nAP_Left,SUB20,6\nAP_Left,SUB21,15\nAP_Left,SUB22,19\nAP_Left,SUB24,12\nAP_Left,SUB26,25\nAP_Right,SUB01,7\nAP_Right,SUB02,20\nAP_Right,SUB03,8\nAP_Right,SUB06,22\nAP_Right,SUB07,16\nAP_Right,SUB10,16\nAP_Right,SUB11,8\nAP_Right,SUB12,15\nAP_Right,SUB13,24\nAP_Right,SUB14,13\nAP_Right,SUB16,13\nAP_Right,SUB17,26\nAP_Right,SUB18,11\nAP_Right,SUB19,13\nAP_Right,SUB20,1\nAP_Right,SUB21,14\nAP_Right,SUB22,17\nAP_Right,SUB24,7\nAP_Right,SUB26,12\nVertical_Left,SUB01,7\nVertical_Left,SUB02,23\nVertical_Left,SUB03,9\nVertical_Left,SUB05,2\nVertical_Left,SUB06,21\nVertical_Left,SUB07,1\nVertical_Left,SUB10,20\nVertical_Left,SUB11,19\nVertical_Left,SUB12,8\nVertical_Left,SUB13,12\nVertical_Left,SUB14,21\nVertical_Left,SUB16,14\nVertical_Left,SUB17,11\nVertical_Left,SUB18,20\nVertical_Left,SUB19,18\nVertical_Left,SUB20,6\nVertical_Left,SUB21,15\nVertical_Left,SUB22,19\nVertical_Left,SUB24,12\nVertical_Left,SUB26,25\nVertical_Right,SUB01,7\nVertical_Right,SUB02,20\nVertical_Right,SUB03,8\nVertical_Right,SUB06,22\nVertical_Right,SUB07,16\nVertical_Right,SUB10,16\nVertical_Right,SUB11,8\nVertical_Right,SUB12,15\nVertical_Right,SUB13,24\nVertical_Right,SUB14,13\nVertical_Right,SUB16,13\nVertical_Right,SUB17,26\nVertical_Right,SUB18,11\nVertical_Right,SUB19,13\nVertical_Right,SUB20,1\nVertical_Right,SUB21,14\nVertical_Right,SUB22,17\nVertical_Right,SUB24,7\nVertical_Right,SUB26,12\nML_Left,SUB01,7\nML_Left,SUB02,23\nML_Left,SUB03,9\nML_Left,SUB05,2\nML_Left,SUB06,21\nML_Left,SUB07,1\nML_Left,SUB10,20\nML_Left,SUB11,19\nML_Left,SUB12,8\nML_Left,SUB13,12\nML_Left,SUB14,21\nML_Left,SUB16,14\nML_Left,SUB17,11\nML_Left,SUB18,20\nML_Left,SUB19,18\nML_Left,SUB20,6\nML_Left,SUB21,15\nML_Left,SUB22,19\nML_Left,SUB24,12\nML_Left,SUB26,25\nML_Right,SUB01,7\nML_Right,SUB02,20\nML_Right,SUB03,8\nML_Right,SUB06,22\nML_Right,SUB07,16\nML_Right,SUB10,16\nML_Right,SUB11,8\nML_Right,SUB12,15\nML_Right,SUB13,24\nML_Right,SUB14,13\nML_Right,SUB16,13\nML_Right,SUB17,26\nML_Right,SUB18,11\nML_Right,SUB19,13\nML_Right,SUB20,1\nML_Right,SUB21,14\nML_Right,SUB22,17\nML_Right,SUB24,7\nML_Right,SUB26,12\n'
from io import StringIO
internal = pd.read_excel(SOURCES / 'PDGinfo_Shida.xlsx', sheet_name='PDGinfo')
internal = internal.dropna(how='all').copy()
internal.columns = internal.columns.str.strip()
internal['ID'] = internal.ID.astype(str).str.strip()
if not internal.ID.str.fullmatch(r'SUB\d+').all() or internal.ID.duplicated().any():
    raise ValueError('Unexpected or duplicate internal participant IDs')
external = pd.read_excel(SOURCES / 'Partecipants_Viglialoro.xlsx', sheet_name='Partecipants').dropna(how='all')
external.columns = external.columns.str.strip()
for column in external.select_dtypes(include='object'):
    external[column] = external[column].map(lambda x: x.replace('\xa0', ' ').strip() if isinstance(x, str) else x)
for column in ['Patient ID', 'Age (y)', 'Bodymass (kg)', 'Height (m)', 'BMI']:
    external[column] = pd.to_numeric(external[column].astype(str).str.replace(',', '.', regex=False), errors='raise')
if external['Patient ID'].duplicated().any():
    raise ValueError('Duplicate external participant IDs')
audit = pd.read_csv(StringIO(WAVEFORM_AUDIT_CSV))
available = set(audit.subject)
coverage = pd.DataFrame({'participant_id': internal.ID})
coverage['available_in_supplied_waveforms'] = coverage.participant_id.isin(available)
internal.to_csv(ROOT / 'internal_clinical_records.csv', index=False)
external.to_csv(ROOT / 'external_demographic_records.csv', index=False)
coverage.to_csv(ROOT / 'internal_metadata_coverage.csv', index=False)
display(pd.DataFrame([
    {'cohort': 'Shida internal', 'metadata_participants': len(internal),
     'waveform_participants': len(available), 'test_p_values_supplied': False},
    {'cohort': 'Viglialoro external', 'metadata_participants': len(external),
     'waveform_participants': 'Column-to-participant mapping not supplied', 'test_p_values_supplied': False}
]))
print('Clinical-only internal IDs:', ', '.join(sorted(set(internal.ID) - available)))
print('Left tasks have 20 participants; right tasks have 19. Use test-specific sample sizes.')
print('External demographics now provide body mass and height, but the 23 waveform columns still require identity and visit mapping. Do not map by row order.')
print('Original clinical columns:')
display(pd.DataFrame({'column': internal.columns}))

## Supply the complete test tables

The templates below contain **no statistical results**. Fill in the original hypothesis labels and full-precision p-values. Export numbers, not significance stars or strings such as `<0.001`. A printed `0.0000` must be replaced by the unrounded original value.

**`clinical_correlations.csv` — 396 rows.** Required: `test_id`, `error_metric`, `clinical_measure`, `p_raw`. Each `error_metric` identifies one architecture × side × component, for example `lstm_AP_Left`. There must be all 18 × 22 distinct pairs. Recommended: `r`, `n_participants`, `test_method`, and `medication_state`.

**`gait_comparisons.csv` — 19 rows.** Required: `test_id`, `gait_parameter`, `p_raw`. One original omnibus gait test per parameter. Recommended: `test_statistic`, `effect_size`, `n_participants`, `test_method`, and `medication_state`. Additional post-hoc comparisons require their own declared correction plan.

For recomputation instead of correcting existing p-values, supply participant-level NRMSE for the 18 metrics, the exact 22-variable list with medication state, and participant-level observations for the 19 gait parameters. The participant workbooks alone do not supply these measurements.

Do not interpret the embedded 26 clinical records as 26 participants with available prediction errors. Do not include external demographic associations in these internal families.

In [ ]:
clinical_template = pd.DataFrame({
    'test_id': [f'C{i:03d}' for i in range(1,397)],
    'error_metric': '', 'clinical_measure': '', 'r': np.nan,
    'n_participants': np.nan, 'test_method': '', 'medication_state': '', 'p_raw': np.nan})
gait_template = pd.DataFrame({
    'test_id': [f'G{i:02d}' for i in range(1,20)],
    'gait_parameter': '', 'test_statistic': np.nan, 'effect_size': np.nan,
    'n_participants': np.nan, 'test_method': '', 'medication_state': '', 'p_raw': np.nan})
for name, frame in [('clinical_correlations.csv', clinical_template), ('gait_comparisons.csv', gait_template)]:
    path = TEMPLATES / name
    if not path.exists():
        frame.to_csv(path, index=False)
print('Blank templates:', TEMPLATES)
print('Save completed input files in:', INPUTS)

In [ ]:
# Optional: download both blank CSV templates.
DOWNLOAD_TEMPLATES = False
if DOWNLOAD_TEMPLATES:
    from google.colab import files
    template_zip = shutil.make_archive('/content/fdr_input_templates', 'zip', TEMPLATES)
    files.download(template_zip)

# Set True to upload completed CSVs from your computer.
UPLOAD_COMPLETED_TABLES = False
if UPLOAD_COMPLETED_TABLES:
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        name = Path(name).name
        if name not in {'clinical_correlations.csv', 'gait_comparisons.csv'}:
            raise ValueError('Use the exact documented CSV filename')
        destination = INPUTS / name
        if destination.exists() and destination.read_bytes() != content:
            backup = INPUTS / (destination.stem + '_previous_' + hashlib.sha256(destination.read_bytes()).hexdigest()[:12] + '.csv')
            if not backup.exists():
                backup.write_bytes(destination.read_bytes())
        destination.write_bytes(content)
    print('Input files saved. Previous differing inputs are retained as backups.')

In [ ]:
"""Correct complete internal-cohort hypothesis families without selecting by p-value."""
import numpy as np
import pandas as pd
from scipy.stats import false_discovery_control

FAMILIES = {"clinical_correlations": 396, "gait_comparisons": 19}

def correct_family(frame, family, alpha=0.05):
    if family not in FAMILIES:
        raise ValueError("Unknown hypothesis family")
    expected = FAMILIES[family]
    if len(frame) != expected:
        raise ValueError(f"{family}: require all {expected} tests, including nonsignificant tests; received {len(frame)}")
    required = ["test_id", "p_raw"]
    labels = ["error_metric", "clinical_measure"] if family == "clinical_correlations" else ["gait_parameter"]
    required += labels
    absent = sorted(set(required) - set(frame.columns))
    if absent:
        raise ValueError(f"Missing columns: {absent}")
    result = frame.copy()
    for column in ["test_id", *labels]:
        result[column] = result[column].astype("string").str.strip()
        if result[column].isna().any() or result[column].eq("").any():
            raise ValueError(f"Provide every {column}; do not invent hypothesis labels")
    if result.test_id.duplicated().any() or result.duplicated(labels).any():
        raise ValueError("Duplicate tests: supply each hypothesis once")
    if "cohort" in result:
        cohorts = result.cohort.astype("string").str.strip().str.lower()
        if cohorts.isna().any() or not cohorts.isin(["internal", "shida"]).all():
            raise ValueError("These two correction families are for the Shida internal cohort only")
    if family == "clinical_correlations":
        if result.error_metric.nunique() != 18 or result.clinical_measure.nunique() != 22:
            raise ValueError("Require exactly 18 error metrics x 22 clinical measures, one row per combination")
    p = pd.to_numeric(result.p_raw, errors="coerce").to_numpy(dtype=float)
    if not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
        raise ValueError("Every p_raw must be an exact numeric p-value in [0,1]; blanks, '<0.05' and '<0.001' are not accepted")
    if np.any(p == 0):
        raise ValueError("p_raw contains zero. Export full-precision p-values; investigate rounding or numerical underflow before correction")
    for column in ("n_participants", "r"):
        if column in result:
            raw = result[column]
            provided = raw.notna() & raw.astype(str).str.strip().ne("")
            values = pd.to_numeric(raw, errors="coerce")
            if not np.isfinite(values[provided]).all():
                raise ValueError(f"Invalid numeric {column}")
            if column == "n_participants" and ((values[provided] < 3) | (values[provided] % 1 != 0)).any():
                raise ValueError("n_participants must contain integer participant counts >=3")
            if column == "r" and (values[provided].abs() > 1).any():
                raise ValueError("Correlation coefficients must be between -1 and 1")
    result["p_raw"] = p
    result["family"] = family
    result["family_size"] = expected
    result["q_bh"] = false_discovery_control(p, method="bh")
    result["q_by"] = false_discovery_control(p, method="by")
    result["nominal_p_lt_0_05"] = p < alpha
    result["bh_q_lt_0_05"] = result.q_bh < alpha
    result["by_q_lt_0_05"] = result.q_by < alpha
    return result

def summarize(result):
    return {
        "family": result.family.iloc[0],
        "n_tests": len(result),
        "nominal_p_lt_0_05": int(result.nominal_p_lt_0_05.sum()),
        "bh_q_lt_0_05": int(result.bh_q_lt_0_05.sum()),
        "by_q_lt_0_05": int(result.by_q_lt_0_05.sum()),
        "smallest_p": float(result.p_raw.min()),
        "smallest_q_bh": float(result.q_bh.min()),
        "smallest_q_by": float(result.q_by.min()),
    }

## Apply correction and export

Choose a label that distinguishes the original manuscript from a newly recomputed nested-validation analysis. The output folder includes hashes of the input tables and correction code, preserving previous results. Missing or invalid inputs yield a pending/invalid status, not an inferred count of significant findings.

In [ ]:
CORRECTION_CODE_SHA256 = '2ddafbb67266e4d02dcda15669f95158dafe764f4fc9089ec97a779a5c891a88'
ANALYSIS_LABEL = 'original_manuscript'
if not re.fullmatch(r'[A-Za-z0-9_-]+', ANALYSIS_LABEL):
    raise ValueError('Use only letters, digits, underscores and hyphens in ANALYSIS_LABEL')
paths = {family: INPUTS / f'{family}.csv' for family in FAMILIES}
input_hashes = {family: hashlib.sha256(path.read_bytes()).hexdigest() if path.exists() else None
                for family, path in paths.items()}
identity = {'label': ANALYSIS_LABEL, 'input_sha256': input_hashes,
            'correction_code_sha256': CORRECTION_CODE_SHA256,
            'scipy': scipy.__version__, 'alpha': 0.05, 'methods': ['bh', 'by']}
run_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
OUTPUT = ROOT / 'results' / f'{ANALYSIS_LABEL}_{run_id}'
OUTPUT.mkdir(parents=True, exist_ok=True)
statuses, summaries, wording = [], [], []
for family, path in paths.items():
    if not path.exists():
        statuses.append({'family': family, 'status': 'pending', 'reason': f'Missing {path.name}'})
        continue
    try:
        result = correct_family(pd.read_csv(path), family)
    except (ValueError, pd.errors.ParserError, pd.errors.EmptyDataError) as error:
        statuses.append({'family': family, 'status': 'invalid input', 'reason': str(error)})
        continue
    result.to_csv(OUTPUT / f'{family}_all_q_values.csv', index=False, float_format='%.17g')
    result[result.bh_q_lt_0_05].to_csv(OUTPUT / f'{family}_bh_discoveries.csv', index=False, float_format='%.17g')
    result[result.by_q_lt_0_05].to_csv(OUTPUT / f'{family}_by_discoveries.csv', index=False, float_format='%.17g')
    shutil.copy2(path, OUTPUT / ('input_' + path.name))
    summary = summarize(result)
    summaries.append(summary)
    statuses.append({'family': family, 'status': 'completed', 'reason': 'All tests included; BH and BY both reported'})
    noun = 'clinical correlations' if family == 'clinical_correlations' else 'gait-parameter comparisons'
    wording.append(f"Among {summary['n_tests']} {noun}, {summary['nominal_p_lt_0_05']} had unadjusted p < 0.05, "
                   f"{summary['bh_q_lt_0_05']} met BH-adjusted q < 0.05, and {summary['by_q_lt_0_05']} met BY-adjusted q < 0.05. "
                   'These analyses were exploratory.')
status = pd.DataFrame(statuses)
status.to_csv(OUTPUT / 'correction_status.csv', index=False)
display(status)
if summaries:
    summary_table = pd.DataFrame(summaries)
    summary_table.to_csv(OUTPUT / 'fdr_summary.csv', index=False)
    display(summary_table)
    (OUTPUT / 'results_wording.txt').write_text('\n\n'.join(wording), encoding='utf-8')
    print('\n\n'.join(wording))
else:
    print('No corrected significance counts can be reported yet.')
provenance = {**identity, 'run_utc': datetime.now(timezone.utc).isoformat(),
              'python': platform.python_version(), 'pandas': pd.__version__,
              'participant_workbook_sha256': {k: v['sha256'] for k,v in EMBEDDED.items()},
              'source': 'https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.false_discovery_control.html',
              'clinical_status': 'Participant-level test validity is not established by p-value adjustment alone'}
(OUTPUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
print('Results folder:', OUTPUT)

## Manuscript revision

Remove the abstract's claim of **“62 significant correlations”** unless the revised statement clearly identifies unadjusted exploratory results. Once complete q-values are available, report the corrected count, method, effect sizes and test-specific participant counts. No corrected count can be inferred from 62/396 alone.

Until correction is complete, a factual replacement is: “Associations between participant-level prediction errors and clinical characteristics were explored.” Do not claim that correction has already been performed.

Update the statistical methods, abstract, clinical-results paragraph, gait-results paragraph, discussion, Figure 4/Figure 7 significance markings and supplementary tables consistently. Report every tested relationship, including nonsignificant ones. Do not infer a clinical biomarker from a correlation between disease severity and model error.

The gait family is separate from the correlation family because it addresses a different question. Do not split either family by model, component or disease score to obtain more favorable adjusted values.

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    zip_path = shutil.make_archive('/content/' + OUTPUT.name, 'zip', OUTPUT)
    files.download(zip_path)